# exp000 詳細分析レポート

このノートブックでは、exp000（ConvNeXt Tiny + 5-fold CV）の結果について包括的な分析を実施します。

## 実験概要
- **モデル**: ConvNeXt Tiny (pretrained on ImageNet-22k)
- **タスク**: 牧草地画像から5つのバイオマスターゲットを予測
- **データ**: 357サンプル、5-fold CV
- **総合スコア**: Mean R2 = 0.520 (std = 0.080)

## 1. セットアップとデータ読み込み

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from scipy import stats
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# プロット設定
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# データ読み込み
train_df = pd.read_csv("../../output/exp000/preprocessed_train.csv")
oofs_df = pd.read_csv("../../output/exp000/oofs.csv")

with open("../../output/exp000/results.json", "r") as f:
    results = json.load(f)

# ターゲット列の定義
target_columns = [
    "clover_target",
    "dead_target",
    "green_target",
    "gdm_target",
    "total_target",
]

# ウェイトの定義（競技メトリクス）
target_weights = {
    "clover_target": 0.1,
    "dead_target": 0.1,
    "green_target": 0.1,
    "gdm_target": 0.2,
    "total_target": 0.5,
}

# ターゲット名のマッピング（表示用）
target_names = {
    "clover_target": "Dry Clover",
    "dead_target": "Dry Dead",
    "green_target": "Dry Green",
    "gdm_target": "GDM",
    "total_target": "Dry Total",
}

print(f"Train data shape: {train_df.shape}")
print(f"OOF predictions shape: {oofs_df.shape}")
print(f"\nFold scores: {results['fold_scores']}")
print(f"Mean score: {results['mean_score']:.4f}")
print(f"Std score: {results['std_score']:.4f}")

In [ ]:
# データの基本統計
print("\n=== Train Data Statistics ===")
print(train_df[target_columns].describe())

print("\n=== OOF Predictions Statistics ===")
print(oofs_df[target_columns].describe())

## 2. パフォーマンス詳細分析

In [ ]:
# Fold別スコアの可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 棒グラフ
fold_scores = results['fold_scores']
fold_names = [f'Fold {i}' for i in range(len(fold_scores))]
colors = ['#FF6B6B' if score < 0.5 else '#4ECDC4' if score < 0.6 else '#95E1D3' for score in fold_scores]

axes[0].bar(fold_names, fold_scores, color=colors, alpha=0.7, edgecolor='black')
axes[0].axhline(y=results['mean_score'], color='red', linestyle='--', label=f'Mean: {results["mean_score"]:.4f}')
axes[0].axhline(y=results['mean_score'] + results['std_score'], color='orange', linestyle=':', alpha=0.5)
axes[0].axhline(y=results['mean_score'] - results['std_score'], color='orange', linestyle=':', alpha=0.5)
axes[0].set_ylabel('Weighted R2 Score')
axes[0].set_title('Fold-wise Performance')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 箱ひげ図
axes[1].boxplot(fold_scores, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].scatter([1]*len(fold_scores), fold_scores, c='blue', s=100, alpha=0.6, zorder=3)
axes[1].set_ylabel('Weighted R2 Score')
axes[1].set_xticklabels(['All Folds'])
axes[1].set_title('Score Distribution Across Folds')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n**Key Findings:**")
print(f"- Best fold: Fold {np.argmax(fold_scores)} (score: {max(fold_scores):.4f})")
print(f"- Worst fold: Fold {np.argmin(fold_scores)} (score: {min(fold_scores):.4f})")
print(f"- Fold variance: {np.var(fold_scores):.6f}")
print(f"- Score range: {max(fold_scores) - min(fold_scores):.4f}")

In [ ]:
# ターゲット別のR2スコア計算
target_r2_scores = {}
target_mae_scores = {}
target_rmse_scores = {}
weighted_contributions = {}

for target in target_columns:
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    target_r2_scores[target] = r2
    target_mae_scores[target] = mae
    target_rmse_scores[target] = rmse
    weighted_contributions[target] = r2 * target_weights[target]

# 結果の可視化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# R2スコア
ax = axes[0, 0]
targets = [target_names[t] for t in target_columns]
r2_values = [target_r2_scores[t] for t in target_columns]
colors_r2 = ['#FF6B6B' if r2 < 0.3 else '#FFD93D' if r2 < 0.5 else '#6BCF7F' for r2 in r2_values]
bars = ax.barh(targets, r2_values, color=colors_r2, alpha=0.7, edgecolor='black')
ax.set_xlabel('R2 Score')
ax.set_title('Target-wise R2 Scores')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.grid(axis='x', alpha=0.3)
for i, (bar, val) in enumerate(zip(bars, r2_values)):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
            va='center', fontsize=9, fontweight='bold')

# Weighted Contribution
ax = axes[0, 1]
weighted_values = [weighted_contributions[t] for t in target_columns]
ax.barh(targets, weighted_values, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Weighted R2 Contribution')
ax.set_title('Target-wise Weighted R2 Contributions')
ax.grid(axis='x', alpha=0.3)
for i, (target, val) in enumerate(zip(targets, weighted_values)):
    weight = target_weights[target_columns[i]]
    ax.text(val + 0.005, i, f'{val:.3f} (w={weight})', 
            va='center', fontsize=8)

# MAE
ax = axes[1, 0]
mae_values = [target_mae_scores[t] for t in target_columns]
ax.barh(targets, mae_values, color='coral', alpha=0.7, edgecolor='black')
ax.set_xlabel('MAE (g/m²)')
ax.set_title('Target-wise Mean Absolute Error')
ax.grid(axis='x', alpha=0.3)

# RMSE
ax = axes[1, 1]
rmse_values = [target_rmse_scores[t] for t in target_columns]
ax.barh(targets, rmse_values, color='mediumpurple', alpha=0.7, edgecolor='black')
ax.set_xlabel('RMSE (g/m²)')
ax.set_title('Target-wise Root Mean Squared Error')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# サマリーテーブル
metrics_df = pd.DataFrame({
    'Target': [target_names[t] for t in target_columns],
    'R2': [target_r2_scores[t] for t in target_columns],
    'MAE': [target_mae_scores[t] for t in target_columns],
    'RMSE': [target_rmse_scores[t] for t in target_columns],
    'Weight': [target_weights[t] for t in target_columns],
    'Weighted_R2': [weighted_contributions[t] for t in target_columns],
})
print("\n=== Target-wise Performance Summary ===")
print(metrics_df.to_string(index=False))
print(f"\nSum of Weighted R2: {sum(weighted_contributions.values()):.4f}")

## 3. 予測精度の可視化

In [ ]:
# 予測値 vs 実測値の散布図（5ターゲット）
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    
    # Fold別に色分け
    for fold in range(5):
        mask = train_df['fold'] == fold
        ax.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=50, label=f'Fold {fold}')
    
    # 対角線（完全予測線）
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
    
    ax.set_xlabel('True Value (g/m²)', fontsize=10)
    ax.set_ylabel('Predicted Value (g/m²)', fontsize=10)
    ax.set_title(f'{target_names[target]}\nR² = {target_r2_scores[target]:.3f}, MAE = {target_mae_scores[target]:.2f}', 
                 fontsize=11, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)

# 最後のサブプロットを削除
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
# Residual plots（残差プロット）
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    residuals = y_true - y_pred
    
    # 残差 vs 予測値
    ax.scatter(y_pred, residuals, alpha=0.5, s=30, c='steelblue')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    
    # 平均と標準偏差
    mean_res = np.mean(residuals)
    std_res = np.std(residuals)
    ax.axhline(y=mean_res, color='green', linestyle=':', linewidth=1, label=f'Mean: {mean_res:.2f}')
    ax.axhline(y=mean_res + 2*std_res, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    ax.axhline(y=mean_res - 2*std_res, color='orange', linestyle=':', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Predicted Value (g/m²)', fontsize=10)
    ax.set_ylabel('Residual (True - Pred)', fontsize=10)
    ax.set_title(f'{target_names[target]} Residuals\nMean: {mean_res:.2f}, Std: {std_res:.2f}', 
                 fontsize=11, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

In [ ]:
# QQプロット（正規性確認）
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    residuals = y_true - y_pred
    
    # QQプロット
    stats.probplot(residuals, dist="norm", plot=ax)
    ax.set_title(f'{target_names[target]} Q-Q Plot', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

print("\n**Normality Test (Shapiro-Wilk):**")
for target in target_columns:
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    residuals = y_true - y_pred
    stat, p_value = stats.shapiro(residuals)
    print(f"{target_names[target]}: statistic={stat:.4f}, p-value={p_value:.6f} {'(Normal)' if p_value > 0.05 else '(Not Normal)'}")

## 4. 物理制約の検証

理論上、`Dry_Total_g = Dry_Clover_g + Dry_Dead_g + Dry_Green_g` であるべきです。
モデルが各ターゲットを独立に予測しているため、この制約が守られているか検証します。

In [ ]:
# 物理制約の検証
component_cols = ["clover_target", "dead_target", "green_target"]

# Train データ
train_sum = train_df[component_cols].sum(axis=1)
train_total = train_df["total_target"]
train_violation = train_total - train_sum

# OOF 予測
oof_sum = oofs_df[component_cols].sum(axis=1)
oof_total = oofs_df["total_target"]
oof_violation = oof_total - oof_sum

# 統計量
print("\n=== Physical Constraint Violation Analysis ===")
print("\nTrain Data (Ground Truth):")
print(f"  Mean violation: {train_violation.mean():.6f}")
print(f"  Std violation: {train_violation.std():.6f}")
print(f"  Max violation: {train_violation.abs().max():.6f}")
print(f"  Samples with violation > 0.01: {(train_violation.abs() > 0.01).sum()}")

print("\nOOF Predictions:")
print(f"  Mean violation: {oof_violation.mean():.4f}")
print(f"  Std violation: {oof_violation.std():.4f}")
print(f"  Max violation: {oof_violation.abs().max():.4f}")
print(f"  MAE violation: {np.abs(oof_violation).mean():.4f}")
print(f"  RMSE violation: {np.sqrt(np.mean(oof_violation**2)):.4f}")

# 可視化
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Total vs Sum 散布図（Train）
ax = axes[0, 0]
ax.scatter(train_sum, train_total, alpha=0.5, s=30, c='blue', label='Train')
min_val = min(train_sum.min(), train_total.min())
max_val = max(train_sum.max(), train_total.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
ax.set_xlabel('Sum of Components (g/m²)')
ax.set_ylabel('Total Target (g/m²)')
ax.set_title('Train: Total vs Sum of Components')
ax.legend()
ax.grid(alpha=0.3)

# Total vs Sum 散布図（OOF）
ax = axes[0, 1]
ax.scatter(oof_sum, oof_total, alpha=0.5, s=30, c='green', label='OOF')
min_val = min(oof_sum.min(), oof_total.min())
max_val = max(oof_sum.max(), oof_total.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
ax.set_xlabel('Sum of Predicted Components (g/m²)')
ax.set_ylabel('Predicted Total (g/m²)')
ax.set_title('OOF: Total vs Sum of Components')
ax.legend()
ax.grid(alpha=0.3)

# Violation 分布（ヒストグラム）
ax = axes[1, 0]
ax.hist(train_violation, bins=50, alpha=0.5, color='blue', label='Train', edgecolor='black')
ax.hist(oof_violation, bins=50, alpha=0.5, color='green', label='OOF', edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Violation (Total - Sum)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Physical Constraint Violation')
ax.legend()
ax.grid(alpha=0.3)

# Violation の絶対値（箱ひげ図）
ax = axes[1, 1]
box_data = [train_violation.abs(), oof_violation.abs()]
bp = ax.boxplot(box_data, labels=['Train', 'OOF'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightgreen')
ax.set_ylabel('Absolute Violation (g/m²)')
ax.set_title('Absolute Physical Constraint Violation')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Violation が大きいサンプルの特定
top_violation_indices = oof_violation.abs().nlargest(10).index

print("\n=== Top 10 Samples with Largest Constraint Violation ===")
violation_analysis = train_df.loc[top_violation_indices, ['sample_id', 'state', 'species', 'fold']].copy()
violation_analysis['true_total'] = train_df.loc[top_violation_indices, 'total_target'].values
violation_analysis['true_sum'] = train_sum.loc[top_violation_indices].values
violation_analysis['pred_total'] = oofs_df.loc[top_violation_indices, 'total_target'].values
violation_analysis['pred_sum'] = oof_sum.loc[top_violation_indices].values
violation_analysis['violation'] = oof_violation.loc[top_violation_indices].values
print(violation_analysis.to_string(index=False))

## 5. エラー分析

In [ ]:
# 各ターゲットで誤差の大きいサンプルの特定
print("\n=== Top 10 Samples with Largest Errors (by Target) ===")

for target in target_columns:
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    errors = np.abs(y_true - y_pred)
    
    top_error_indices = np.argsort(errors)[-10:][::-1]
    
    print(f"\n### {target_names[target]} ###")
    error_df = train_df.iloc[top_error_indices][['sample_id', 'state', 'species', 'fold']].copy()
    error_df['true'] = y_true[top_error_indices]
    error_df['pred'] = y_pred[top_error_indices]
    error_df['error'] = errors[top_error_indices]
    error_df['pct_error'] = (errors[top_error_indices] / (y_true[top_error_indices] + 1e-8)) * 100
    print(error_df.to_string(index=False))

In [ ]:
# 負の予測値の分析
print("\n=== Negative Predictions Analysis ===")

for target in target_columns:
    y_pred = oofs_df[target].values
    negative_mask = y_pred < 0
    n_negative = negative_mask.sum()
    
    if n_negative > 0:
        print(f"\n{target_names[target]}:")
        print(f"  Number of negative predictions: {n_negative} ({n_negative/len(y_pred)*100:.2f}%)")
        print(f"  Min prediction: {y_pred.min():.4f}")
        print(f"  Mean of negative predictions: {y_pred[negative_mask].mean():.4f}")
        
        # 負の予測を持つサンプルの特徴
        if n_negative <= 10:
            neg_df = train_df[negative_mask][['sample_id', 'state', 'species']].copy()
            neg_df['true'] = train_df[negative_mask][target].values
            neg_df['pred'] = y_pred[negative_mask]
            print(neg_df.to_string(index=False))
    else:
        print(f"\n{target_names[target]}: No negative predictions")

In [ ]:
# エラーの共通特性分析（State, Species別）
# 各ターゲットでエラーが大きいサンプルの State/Species 分布
fig, axes = plt.subplots(3, 2, figsize=(15, 15))

for idx, target in enumerate(target_columns):
    y_true = train_df[target].values
    y_pred = oofs_df[target].values
    errors = np.abs(y_true - y_pred)
    
    # 上位25%のエラー
    error_threshold = np.percentile(errors, 75)
    high_error_mask = errors >= error_threshold
    
    # State 分布
    ax = axes[idx, 0]
    state_counts = train_df[high_error_mask]['state'].value_counts()
    ax.bar(state_counts.index, state_counts.values, color='salmon', alpha=0.7, edgecolor='black')
    ax.set_xlabel('State')
    ax.set_ylabel('Count')
    ax.set_title(f'{target_names[target]}: State Distribution (High Error Samples)')
    ax.grid(axis='y', alpha=0.3)
    
    # Species 分布（上位10種）
    ax = axes[idx, 1]
    species_counts = train_df[high_error_mask]['species'].value_counts().head(10)
    ax.barh(range(len(species_counts)), species_counts.values, color='skyblue', alpha=0.7, edgecolor='black')
    ax.set_yticks(range(len(species_counts)))
    ax.set_yticklabels(species_counts.index, fontsize=8)
    ax.set_xlabel('Count')
    ax.set_title(f'{target_names[target]}: Top 10 Species (High Error Samples)')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. データ特性別のパフォーマンス

In [ ]:
# State別のパフォーマンス
states = train_df['state'].unique()
state_performance = {}

for state in states:
    state_mask = train_df['state'] == state
    state_metrics = {}
    
    for target in target_columns:
        y_true = train_df.loc[state_mask, target].values
        y_pred = oofs_df.loc[state_mask, target].values
        
        if len(y_true) > 0:
            r2 = r2_score(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            state_metrics[target] = {'r2': r2, 'mae': mae, 'count': len(y_true)}
    
    state_performance[state] = state_metrics

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R2スコア（State別）
ax = axes[0]
state_r2_data = []
for state in states:
    r2_values = [state_performance[state][t]['r2'] for t in target_columns]
    state_r2_data.append(r2_values)

x = np.arange(len(target_columns))
width = 0.2
for i, state in enumerate(states):
    ax.bar(x + i*width, state_r2_data[i], width, label=state, alpha=0.7)

ax.set_xlabel('Target')
ax.set_ylabel('R2 Score')
ax.set_title('R2 Score by State')
ax.set_xticks(x + width * (len(states)-1) / 2)
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0, color='black', linewidth=0.5)

# MAE（State別）
ax = axes[1]
state_mae_data = []
for state in states:
    mae_values = [state_performance[state][t]['mae'] for t in target_columns]
    state_mae_data.append(mae_values)

for i, state in enumerate(states):
    ax.bar(x + i*width, state_mae_data[i], width, label=state, alpha=0.7)

ax.set_xlabel('Target')
ax.set_ylabel('MAE (g/m²)')
ax.set_title('MAE by State')
ax.set_xticks(x + width * (len(states)-1) / 2)
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# サンプル数
print("\n=== Sample Counts by State ===")
print(train_df['state'].value_counts())

In [ ]:
# Species別のパフォーマンス（上位10種）
top_species = train_df['species'].value_counts().head(10).index
species_performance = {}

for species in top_species:
    species_mask = train_df['species'] == species
    species_metrics = {}
    
    for target in target_columns:
        y_true = train_df.loc[species_mask, target].values
        y_pred = oofs_df.loc[species_mask, target].values
        
        if len(y_true) > 1:
            r2 = r2_score(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            species_metrics[target] = {'r2': r2, 'mae': mae, 'count': len(y_true)}
    
    species_performance[species] = species_metrics

# 可視化（ヒートマップ）
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# R2スコアのヒートマップ
r2_matrix = []
for species in top_species:
    r2_values = [species_performance[species].get(t, {}).get('r2', np.nan) for t in target_columns]
    r2_matrix.append(r2_values)

ax = axes[0]
im = ax.imshow(r2_matrix, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=1.0)
ax.set_xticks(range(len(target_columns)))
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.set_yticks(range(len(top_species)))
ax.set_yticklabels(top_species, fontsize=9)
ax.set_title('R2 Score by Species (Top 10)')
plt.colorbar(im, ax=ax)

# 値を表示
for i in range(len(top_species)):
    for j in range(len(target_columns)):
        if not np.isnan(r2_matrix[i][j]):
            text = ax.text(j, i, f'{r2_matrix[i][j]:.2f}',
                          ha="center", va="center", color="black", fontsize=8)

# MAEのヒートマップ
mae_matrix = []
for species in top_species:
    mae_values = [species_performance[species].get(t, {}).get('mae', np.nan) for t in target_columns]
    mae_matrix.append(mae_values)

ax = axes[1]
im = ax.imshow(mae_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(target_columns)))
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.set_yticks(range(len(top_species)))
ax.set_yticklabels(top_species, fontsize=9)
ax.set_title('MAE by Species (Top 10)')
plt.colorbar(im, ax=ax, label='MAE (g/m²)')

plt.tight_layout()
plt.show()

print("\n=== Sample Counts by Species (Top 10) ===")
print(train_df['species'].value_counts().head(10))

In [ ]:
# 時系列分析（月別）
train_df['sampling_date'] = pd.to_datetime(train_df['sampling_date'], format='%Y/%m/%d')
train_df['month'] = train_df['sampling_date'].dt.month
train_df['quarter'] = train_df['sampling_date'].dt.quarter

# 月別のパフォーマンス（total_targetのみ）
months = sorted(train_df['month'].unique())
monthly_r2 = []
monthly_mae = []
monthly_counts = []

for month in months:
    month_mask = train_df['month'] == month
    y_true = train_df.loc[month_mask, 'total_target'].values
    y_pred = oofs_df.loc[month_mask, 'total_target'].values
    
    if len(y_true) > 1:
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        monthly_r2.append(r2)
        monthly_mae.append(mae)
        monthly_counts.append(len(y_true))
    else:
        monthly_r2.append(np.nan)
        monthly_mae.append(np.nan)
        monthly_counts.append(len(y_true))

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# R2スコア
ax = axes[0]
ax.plot(months, monthly_r2, marker='o', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Month')
ax.set_ylabel('R2 Score')
ax.set_title('Monthly Performance (Dry Total Target)')
ax.grid(alpha=0.3)
ax.set_xticks(months)

# サンプル数（右軸）
ax2 = ax.twinx()
ax2.bar(months, monthly_counts, alpha=0.3, color='orange', label='Sample Count')
ax2.set_ylabel('Sample Count', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

# MAE
ax = axes[1]
ax.plot(months, monthly_mae, marker='s', linewidth=2, markersize=8, color='coral')
ax.set_xlabel('Month')
ax.set_ylabel('MAE (g/m²)')
ax.set_title('Monthly MAE (Dry Total Target)')
ax.grid(alpha=0.3)
ax.set_xticks(months)

plt.tight_layout()
plt.show()

print("\n=== Sample Counts by Month ===")
print(train_df['month'].value_counts().sort_index())

In [ ]:
# Fold別のデータ分布
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# State分布
ax = axes[0, 0]
fold_state_counts = train_df.groupby(['fold', 'state']).size().unstack(fill_value=0)
fold_state_counts.plot(kind='bar', stacked=True, ax=ax, alpha=0.7)
ax.set_xlabel('Fold')
ax.set_ylabel('Count')
ax.set_title('State Distribution by Fold')
ax.legend(title='State', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Species分布（上位5種）
ax = axes[0, 1]
top_5_species = train_df['species'].value_counts().head(5).index
fold_species_counts = train_df[train_df['species'].isin(top_5_species)].groupby(['fold', 'species']).size().unstack(fill_value=0)
fold_species_counts.plot(kind='bar', stacked=True, ax=ax, alpha=0.7)
ax.set_xlabel('Fold')
ax.set_ylabel('Count')
ax.set_title('Species Distribution by Fold (Top 5)')
ax.legend(title='Species', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(axis='y', alpha=0.3)

# ターゲットの平均値（Fold別）
ax = axes[1, 0]
fold_target_means = train_df.groupby('fold')[target_columns].mean()
x = np.arange(len(target_columns))
width = 0.15
for fold in range(5):
    ax.bar(x + fold*width, fold_target_means.iloc[fold], width, label=f'Fold {fold}', alpha=0.7)
ax.set_xlabel('Target')
ax.set_ylabel('Mean Value (g/m²)')
ax.set_title('Mean Target Values by Fold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ターゲットの標準偏差（Fold別）
ax = axes[1, 1]
fold_target_stds = train_df.groupby('fold')[target_columns].std()
for fold in range(5):
    ax.bar(x + fold*width, fold_target_stds.iloc[fold], width, label=f'Fold {fold}', alpha=0.7)
ax.set_xlabel('Target')
ax.set_ylabel('Std Dev (g/m²)')
ax.set_title('Target Std Dev by Fold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 分布分析の拡充

In [ ]:
# Train vs OOF の詳細な分布比較
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    # ヒストグラム
    ax.hist(train_df[target], bins=30, alpha=0.5, color='blue', label='Train (True)', edgecolor='black', density=True)
    ax.hist(oofs_df[target], bins=30, alpha=0.5, color='green', label='OOF (Pred)', edgecolor='black', density=True)
    
    # KDE
    train_df[target].plot.kde(ax=ax, color='blue', linewidth=2)
    oofs_df[target].plot.kde(ax=ax, color='green', linewidth=2)
    
    ax.set_xlabel('Value (g/m²)')
    ax.set_ylabel('Density')
    ax.set_title(f'{target_names[target]} Distribution')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # 統計量を追加
    train_mean = train_df[target].mean()
    train_std = train_df[target].std()
    oof_mean = oofs_df[target].mean()
    oof_std = oofs_df[target].std()
    
    textstr = f'Train: μ={train_mean:.2f}, σ={train_std:.2f}\nOOF: μ={oof_mean:.2f}, σ={oof_std:.2f}'
    ax.text(0.98, 0.97, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

In [ ]:
# 予測範囲の圧縮問題の定量化
print("\n=== Prediction Range Compression Analysis ===")

compression_stats = []
for target in target_columns:
    train_mean = train_df[target].mean()
    train_std = train_df[target].std()
    train_min = train_df[target].min()
    train_max = train_df[target].max()
    train_iqr = train_df[target].quantile(0.75) - train_df[target].quantile(0.25)
    
    oof_mean = oofs_df[target].mean()
    oof_std = oofs_df[target].std()
    oof_min = oofs_df[target].min()
    oof_max = oofs_df[target].max()
    oof_iqr = oofs_df[target].quantile(0.75) - oofs_df[target].quantile(0.25)
    
    std_ratio = oof_std / train_std
    range_ratio = (oof_max - oof_min) / (train_max - train_min)
    iqr_ratio = oof_iqr / train_iqr
    
    compression_stats.append({
        'Target': target_names[target],
        'Train_Std': train_std,
        'OOF_Std': oof_std,
        'Std_Ratio': std_ratio,
        'Train_Range': train_max - train_min,
        'OOF_Range': oof_max - oof_min,
        'Range_Ratio': range_ratio,
        'IQR_Ratio': iqr_ratio,
    })

compression_df = pd.DataFrame(compression_stats)
print(compression_df.to_string(index=False))

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

targets_short = [target_names[t] for t in target_columns]

# Std Ratio
ax = axes[0]
ax.bar(targets_short, compression_df['Std_Ratio'], color='skyblue', alpha=0.7, edgecolor='black')
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='No Compression')
ax.set_ylabel('OOF Std / Train Std')
ax.set_title('Standard Deviation Ratio')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Range Ratio
ax = axes[1]
ax.bar(targets_short, compression_df['Range_Ratio'], color='lightcoral', alpha=0.7, edgecolor='black')
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='No Compression')
ax.set_ylabel('OOF Range / Train Range')
ax.set_title('Range Ratio')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# IQR Ratio
ax = axes[2]
ax.bar(targets_short, compression_df['IQR_Ratio'], color='lightgreen', alpha=0.7, edgecolor='black')
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='No Compression')
ax.set_ylabel('OOF IQR / Train IQR')
ax.set_title('IQR Ratio')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# 外れ値の検出と分析
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    data_to_plot = [train_df[target], oofs_df[target]]
    bp = ax.boxplot(data_to_plot, labels=['Train', 'OOF'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('lightgreen')
    
    ax.set_ylabel('Value (g/m²)')
    ax.set_title(f'{target_names[target]}')
    ax.grid(axis='y', alpha=0.3)

fig.delaxes(axes[5])
plt.suptitle('Outlier Analysis (Box Plots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ターゲット間の相関分析
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Train データの相関
ax = axes[0]
train_corr = train_df[target_columns].corr()
im = ax.imshow(train_corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(target_columns)))
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.set_yticks(range(len(target_columns)))
ax.set_yticklabels([target_names[t] for t in target_columns])
ax.set_title('Train Data Correlation')
plt.colorbar(im, ax=ax)

# 値を表示
for i in range(len(target_columns)):
    for j in range(len(target_columns)):
        text = ax.text(j, i, f'{train_corr.iloc[i, j]:.2f}',
                      ha="center", va="center", color="black", fontsize=10)

# OOF 予測の相関
ax = axes[1]
oof_corr = oofs_df[target_columns].corr()
im = ax.imshow(oof_corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(target_columns)))
ax.set_xticklabels([target_names[t] for t in target_columns], rotation=45, ha='right')
ax.set_yticks(range(len(target_columns)))
ax.set_yticklabels([target_names[t] for t in target_columns])
ax.set_title('OOF Predictions Correlation')
plt.colorbar(im, ax=ax)

# 値を表示
for i in range(len(target_columns)):
    for j in range(len(target_columns)):
        text = ax.text(j, i, f'{oof_corr.iloc[i, j]:.2f}',
                      ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

## 8. 補助ターゲットとの相関分析

In [ ]:
# 補助ターゲット: pre_gshh_ndvi, height_ave_cm
auxiliary_features = ['pre_gshh_ndvi', 'height_ave_cm']

# 相関係数の計算
print("\n=== Correlation with Auxiliary Features ===")

for aux_feat in auxiliary_features:
    print(f"\n### {aux_feat} ###")
    for target in target_columns:
        corr = train_df[[aux_feat, target]].corr().iloc[0, 1]
        print(f"  {target_names[target]}: {corr:.4f}")

In [ ]:
# 散布図（pre_gshh_ndvi）
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    ax.scatter(train_df['pre_gshh_ndvi'], train_df[target], alpha=0.5, s=30, c='blue')
    
    # 線形回帰
    z = np.polyfit(train_df['pre_gshh_ndvi'], train_df[target], 1)
    p = np.poly1d(z)
    x_line = np.linspace(train_df['pre_gshh_ndvi'].min(), train_df['pre_gshh_ndvi'].max(), 100)
    ax.plot(x_line, p(x_line), "r--", linewidth=2)
    
    corr = train_df[['pre_gshh_ndvi', target]].corr().iloc[0, 1]
    ax.set_xlabel('Pre GSHH NDVI')
    ax.set_ylabel(f'{target_names[target]} (g/m²)')
    ax.set_title(f'Correlation: {corr:.4f}')
    ax.grid(alpha=0.3)

fig.delaxes(axes[5])
plt.suptitle('Correlation with Pre GSHH NDVI', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 散布図（height_ave_cm）
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(target_columns):
    ax = axes[idx]
    
    ax.scatter(train_df['height_ave_cm'], train_df[target], alpha=0.5, s=30, c='green')
    
    # 線形回帰
    z = np.polyfit(train_df['height_ave_cm'], train_df[target], 1)
    p = np.poly1d(z)
    x_line = np.linspace(train_df['height_ave_cm'].min(), train_df['height_ave_cm'].max(), 100)
    ax.plot(x_line, p(x_line), "r--", linewidth=2)
    
    corr = train_df[['height_ave_cm', target]].corr().iloc[0, 1]
    ax.set_xlabel('Height Average (cm)')
    ax.set_ylabel(f'{target_names[target]} (g/m²)')
    ax.set_title(f'Correlation: {corr:.4f}')
    ax.grid(alpha=0.3)

fig.delaxes(axes[5])
plt.suptitle('Correlation with Height Average', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 相関行列のヒートマップ（全ターゲット + 補助特徴）
all_features = target_columns + auxiliary_features
corr_matrix = train_df[all_features].corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(all_features)), [target_names.get(f, f) for f in all_features], rotation=45, ha='right')
plt.yticks(range(len(all_features)), [target_names.get(f, f) for f in all_features])
plt.title('Correlation Matrix: Targets + Auxiliary Features', fontsize=14, fontweight='bold')
plt.colorbar(im, label='Correlation')

# 値を表示
for i in range(len(all_features)):
    for j in range(len(all_features)):
        text = plt.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.show()

## 9. 総合的な改善提案

以下、exp000の分析結果に基づく具体的な改善提案をまとめます。

### 9.1 主要な発見事項（Key Findings）

#### パフォーマンス
- **総合スコア**: Mean Weighted R² = 0.520 (std = 0.080)
- **Fold間の大きな分散**: 最良 0.655 vs 最悪 0.426（差: 0.229）
- **ターゲット別性能**:
  - 良好: GDM (R²=0.61), Dry Total (R²=0.55)
  - 不良: Dry Green (R²=0.16), Dry Dead (R²=0.34)

#### 予測範囲の圧縮
- すべてのターゲットで予測範囲が大幅に圧縮されている
- Std比: 0.36～0.59（理想は1.0）
- Range比: 0.21～0.37（理想は1.0）
- 特にDry Greenで顕著（Range比=0.21）

#### 物理制約違反
- OOF予測での制約違反: Mean = 0.37, Std = 6.96, Max = 23.04
- Train（真値）では制約はほぼ守られている（Max violation = 0.31）
- モデルは各ターゲットを独立に予測しているため、物理的整合性がない

#### 負の予測値
- Dry Clover: 2サンプル（0.56%）
- Dry Dead: 1サンプル（0.28%）
- 負の値は物理的に意味がないため、後処理でclipping必要

#### データ特性
- **State別**: Tasが最多（129サンプル）、Fold間でState分布に偏りあり
- **Species別**: パフォーマンスに大きなばらつき（一部の種で極端に低いR²）
- **時系列**: 月別のサンプル数に偏りあり（9月が最多）

#### 補助ターゲット
- `pre_gshh_ndvi`: GDMとの強い正の相関（0.70～0.75）
- `height_ave_cm`: Dry Totalとの中程度の相関（0.50～0.60）
- これらは補助損失（Auxiliary Loss）として活用可能

### 9.2 改善提案（Improvement Proposals）

#### 1. 物理制約の強制（High Priority）
**問題**: `Dry_Total != Dry_Clover + Dry_Dead + Dry_Green`

**解決策**:
- **Option A**: 4ターゲット予測 + 計算による制約
  - モデルは`Dry_Clover`, `Dry_Dead`, `Dry_Green`, `GDM`の4つのみ予測
  - `Dry_Total = Dry_Clover + Dry_Dead + Dry_Green`として計算
  - Lossは5ターゲット全てで計算（Weighted MSE）
  
- **Option B**: 出力層での制約強制
  - Softmax的な変換でコンポーネントの比率を予測
  - `Total`を別途予測し、各コンポーネントを`Total × 比率`で計算

**期待効果**: 制約違反の完全解消、物理的整合性の確保

---

#### 2. 予測範囲圧縮への対処（High Priority）
**問題**: 予測値の分散・範囲が真値より大幅に小さい

**解決策**:
- **損失関数の見直し**:
  - MSEの代わりにHuber Lossや、外れ値に強い損失関数を試す
  - 予測範囲を広げるための損失項を追加（Variance Penalty）
  
- **出力変換**:
  - ターゲットの対数変換（Log1p）を試す
  - Box-Cox変換で正規分布に近づける
  - 出力層の活性化関数を見直し（ReLU → ELU、Softplus等）
  
- **後処理**:
  - 予測値の分散をTrainに合わせて調整（rescaling）
  - Calibration技術の適用

**期待効果**: 予測範囲の拡大、極値の予測精度向上

---

#### 3. Fold分散の削減（Medium Priority）
**問題**: Fold間のスコア差が大きい（std=0.080）

**解決策**:
- **Stratification の改善**:
  - 現在: ターゲット値でStratifiedKFold
  - 追加: State, Species, 月などの複数要素を考慮したStratification
  
- **データ拡張の見直し**:
  - よりアグレッシブな拡張（Mixup, CutMix）
  - ドメイン固有の拡張（色調変換、季節性を考慮した変換）
  
- **アンサンブル**:
  - 異なるseedでの複数学習
  - Fold間のモデルの平均化

**期待効果**: より安定したCV、未知データへの汎化性能向上

---

#### 4. Dry Greenの改善（Medium Priority）
**問題**: Dry GreenのR²が極端に低い（0.16）

**解決策**:
- **ターゲット別の損失重み調整**:
  - Dry Greenの損失重みを動的に増やす（Focal Loss的なアプローチ）
  
- **マルチタスク学習の改善**:
  - ターゲット別に異なるheadを持つ（shared backbone + multiple heads）
  - Dry Greenに特化した特徴抽出層を追加
  
- **データ分析**:
  - Dry Greenが高い/低いサンプルの画像的特徴を分析
  - 特定の種やStateでパフォーマンスが低い場合、その原因を調査

**期待効果**: Dry Greenの精度向上、総合スコアの改善

---

#### 5. 補助ターゲットの活用（Medium Priority）
**問題**: `pre_gshh_ndvi`, `height_ave_cm`が未使用

**解決策**:
- **Auxiliary Loss**:
  - モデルに補助ターゲットも予測させる（Multi-task Learning）
  - Loss = Main Loss + λ × Auxiliary Loss
  - 特にNDVIはGDMと高相関（0.70+）なので有効
  
- **入力特徴として使用**:
  - NDVIとHeightを画像とconcatenateして入力
  - または、別のMLPブランチで処理してfusionする

**期待効果**: 特徴表現の改善、GDMの精度向上

---

#### 6. アーキテクチャの変更（Low-Medium Priority）
**現状**: ConvNeXt Tiny（CNN）

**検討すべき選択肢**:
- **Vision Transformer系**:
  - ViT, Swin Transformer, MaxViT等
  - 既存所感: "全体の雰囲気をみたいのでtransformer系のほうが良いかも"
  - Global contextの捉え方がCNNより優れている可能性
  
- **Hybrid モデル**:
  - ConvNeXt + Transformer（CoAtNet等）
  - Local + Global features の両方を活用
  
- **より大きなモデル**:
  - ConvNeXt Small/Base（現在はTiny）
  - データ数が少ない（357サンプル）のでoverfittingに注意

**期待効果**: 特徴抽出能力の向上、パフォーマンス改善

---

#### 7. TTA・アンサンブルの実装（High Priority）
**現状**: Single model inference

**実装すべき技術**:
- **Test Time Augmentation (TTA)**:
  - Horizontal/Vertical flip
  - Rotation (90°, 180°, 270°)
  - Multi-scale（複数の画像サイズ）
  - 予測の平均化
  
- **アンサンブル**:
  - 5-fold全モデルの平均
  - 異なるアーキテクチャのアンサンブル
  - Weighted averaging（Foldスコアベース）

**期待効果**: 予測の安定性向上、スコア向上（通常+2~5%）

---

#### 8. その他の改善
- **負の値のClipping**: 後処理で`max(0, prediction)`を適用
- **高速化**: Mixed precision trainingは既に実装済み、推論の最適化を検討
- **データ品質**: "0gでラベルミスがある"との指摘 → 該当サンプルの再確認・除外を検討
- **ドメインシフト対策**: State/Species別のドメイン適応技術の適用を検討

### 9.3 Next Steps（優先順位順）

#### Immediate Actions（exp001）
1. ✅ **物理制約の強制**: 4ターゲット予測 + 計算による`Dry_Total`
2. ✅ **負の値のClipping**: 後処理の実装
3. ✅ **TTA の実装**: 基本的なflip/rotationから開始

#### Short-term Experiments（exp002-004）
4. **補助ターゲットのAuxiliary Loss**
5. **損失関数の変更**: Huber Loss, Focal MSE等を試す
6. **ターゲット変換**: Log1p, Box-Cox等を試す

#### Medium-term Experiments（exp005-007）
7. **Vision Transformer**: Swin Transformer, MaxViT等を試す
8. **より大きなモデル**: ConvNeXt Small/Base
9. **アンサンブル**: 複数モデル・複数foldの組み合わせ

#### Long-term Improvements
10. **ドメイン適応**: State/Species固有の学習
11. **Pseudo-labeling**: Test dataでの半教師あり学習（競技ルール確認必要）
12. **外部データ**: 類似データセットでのpre-training

---

### 9.4 期待されるスコア改善

各改善策の期待効果（経験則ベース）:

| 改善策 | 期待スコア向上 | 実装難易度 | 優先度 |
|--------|---------------|-----------|-------|
| 物理制約の強制 | +0.02～0.05 | 低 | High |
| TTA | +0.01～0.03 | 低 | High |
| 補助ターゲット活用 | +0.01～0.03 | 中 | Medium |
| 予測範囲圧縮対策 | +0.03～0.07 | 中～高 | High |
| Vision Transformer | +0.00～0.05 | 中 | Medium |
| アンサンブル | +0.02～0.05 | 低 | High |
| Dry Green改善 | +0.01～0.02 | 中 | Medium |

**保守的な見積もり**: 上記の改善を組み合わせることで、**Mean R² 0.52 → 0.57～0.62**を目指せる可能性あり。

---

## 分析完了

このノートブックでは、exp000の結果について包括的な分析を実施しました。主要な発見事項と具体的な改善提案をまとめましたので、次の実験（exp001以降）に活かしてください。